# Curve di Bézier per il profilo di uno spoiler

Questo notebook sviluppa, dal primo principio fino a un'applicazione qualitativa, l'uso delle curve di Bézier cubiche per descrivere il profilo bidimensionale di uno spoiler automobilistico. Il percorso è organizzato in quattro step, ciascuno costruito sul precedente:

1. **Step 1 — Curva di Bézier cubica di base.** Si definisce la curva a partire dai polinomi di Bernstein, se ne dimostrano le proprietà fondamentali (interpolazione e tangenza agli estremi, inviluppo convesso) e si verifica numericamente il tutto su un esempio.
2. **Step 2 — Profilo dello spoiler.** Due curve di Bézier che condividono gli estremi (bordo d'attacco e bordo di uscita) compongono la sagoma chiusa del profilo: un dorso convesso e un ventre quasi piatto.
3. **Step 3 — Curvatura lungo il profilo.** Si deriva la formula della curvatura per una curva piana parametrica, la si applica al dorso, e si individua il punto dove la curva "gira" più bruscamente — con un collegamento qualitativo al rischio di separazione del flusso.
4. **Step 4 — Flusso semplificato attorno allo spoiler.** Si usa la soluzione analitica classica del flusso potenziale attorno a un cilindro (corrente uniforme + doppietta) come approssimazione qualitativa di come l'aria "vedrebbe" la sagoma.

**Convenzioni di notazione usate in tutto il notebook**: $P_0, P_1, P_2, P_3$ indicano i quattro punti di controllo di una Bézier cubica (nel piano, quindi ciascuno è una coppia $(x,y)$); $t \in [0,1]$ è il parametro lungo la curva (non ha il significato di "tempo fisico", è solo la variabile che percorre la curva da un estremo all'altro); $B(t) = (x(t), y(t))$ indica la curva stessa, e $B'(t)$, $B''(t)$ le sue derivate prima e seconda rispetto a $t$. Nel codice, ogni punto di controllo è rappresentato come una tupla `(x, y)` di due numeri, e una curva campionata è un array NumPy di forma `(n, 2)` (una riga per ciascuno degli `n` punti campionati lungo `t`).

## Perché le curve di Bézier

Le curve di Bézier nascono nell'industria automobilistica: Pierre Bézier le sviluppò alla Renault negli anni '60 per progettare superfici di carrozzeria, e indipendentemente Paul de Casteljau formulò l'algoritmo geometrico equivalente alla Citroën (rimasto interno, non pubblicato subito). Da lì sono diventate lo strumento base della **Computer-Aided Geometric Design (CAGD)**: font vettoriali, animazione, e — quello che interessa qui — la parametrizzazione di superfici aerodinamiche (profili alari, spoiler, pale di turbina).

Il motivo per cui una parametrizzazione di questo tipo è preferibile a una funzione esplicita $y=f(x)$ è duplice:

- **Forme che una funzione non può rappresentare.** Un profilo alare reale ha tangente verticale al bordo d'attacco e, per definizione, il dorso e il ventre insieme formano una curva *chiusa*: nessuna delle due cose è esprimibile come grafico di una funzione $y=f(x)$ a valore singolo. Una curva parametrica $(x(t),y(t))$ non ha questa limitazione.
- **Controllo locale della forma.** Spostare un punto di controllo modifica la curva in modo prevedibile e localizzato (specialmente per curve di grado basso come la cubica), invece di dover ridefinire un'intera funzione. Questo è esattamente ciò che serve per l'ottimizzazione di forma: si perturba leggermente la geometria (spostando pochi punti di controllo) e si ricalcola il flusso, iterando fino a una forma che soddisfa gli obiettivi aerodinamici.

La Bézier cubica usata qui è il caso più semplice; per superfici più complesse o per garantire una continuità più alta tra segmenti adiacenti (curvatura continua, non solo tangente continua) si usano generalizzazioni come le **B-spline** e le **NURBS** (Non-Uniform Rational B-Splines), che sono lo standard de facto in CAD/CAGD industriale e nei tool di ottimizzazione di forma via CFD.

## Step 1 — Curva di Bézier cubica di base

### Definizione tramite i polinomi di Bernstein

Una curva di Bézier di grado $n$ con punti di controllo $P_0, \dots, P_n$ è definita come combinazione dei punti di controllo pesati dai **polinomi di Bernstein** di grado $n$:

$$B(t) = \sum_{i=0}^{n} P_i \, b_{i,n}(t), \qquad b_{i,n}(t) = \binom{n}{i} t^i (1-t)^{n-i}, \qquad t \in [0,1].$$

Per $n=3$ (caso cubico, quello usato in questo notebook) i quattro polinomi di base sono

$$b_{0,3}(t)=(1-t)^3, \quad b_{1,3}(t)=3(1-t)^2 t, \quad b_{2,3}(t)=3(1-t)t^2, \quad b_{3,3}(t)=t^3,$$

e sostituendo nella formula generale si ottiene esattamente la formula usata nel codice:

$$B(t) = (1-t)^3 P_0 + 3(1-t)^2 t\, P_1 + 3(1-t) t^2\, P_2 + t^3 P_3.$$

### Proprietà fondamentali

Tre proprietà dei polinomi di Bernstein spiegano perché i punti di controllo si comportano in modo così intuitivo:

1. **Partizione dell'unità**: $\sum_i b_{i,n}(t) = (t + (1-t))^n = 1$ per ogni $t$ (binomio di Newton). Di conseguenza $B(t)$ è sempre una **combinazione affine** dei punti di controllo (i pesi sommano a 1): la curva è invariante per trasformazioni affini (traslazioni, rotazioni, scalature) applicate ai punti di controllo — trasformare i punti e poi valutare $B(t)$ dà lo stesso risultato di valutare $B(t)$ e poi trasformare la curva.
2. **Non negatività su $[0,1]$**: per $t\in[0,1]$ si ha $b_{i,n}(t)\ge 0$. Insieme alla partizione dell'unità, questo significa che $B(t)$ è sempre una **combinazione convessa** dei punti di controllo: la curva intera giace nell'inviluppo convesso di $P_0,\dots,P_n$ (proprietà dell'**inviluppo convesso**, *convex hull property*).
3. **Interpolazione agli estremi**: a $t=0$ si ha $b_{0,n}(0)=1$ e tutti gli altri $b_{i,n}(0)=0$ (e simmetricamente a $t=1$ solo $b_{n,n}(1)=1$), quindi $B(0)=P_0$ e $B(1)=P_n$ — la curva passa esattamente per il primo e l'ultimo punto di controllo (ma in generale **non** per quelli intermedi).

### Tangenza agli estremi

Vale una regola generale: la derivata di una Bézier di grado $n$ è, a sua volta, una Bézier di grado $n-1$ costruita sulle **differenze successive** dei punti di controllo, scalate per $n$:

$$B'(t) = n \sum_{i=0}^{n-1} (P_{i+1}-P_i)\, b_{i,n-1}(t).$$

Per $n=3$, valutando agli estremi ($b_{0,2}(0)=1$ e $b_{2,2}(1)=1$, tutti gli altri nulli):

$$B'(0) = 3(P_1-P_0), \qquad B'(1) = 3(P_3-P_2).$$

Cioè la tangente in $t=0$ è **parallela** al segmento $P_0P_1$ (stessa direzione, modulo il fattore $3$), e la tangente in $t=1$ è parallela a $P_2P_3$. È proprio questa proprietà a rendere i punti di controllo intuitivi da manipolare a mano: $P_1$ e $P_2$ funzionano come le "maniglie" che orientano la curva in uscita da $P_0$ e in entrata su $P_3$, esattamente come nei tool di disegno vettoriale.

### Un metodo alternativo: l'algoritmo di de Casteljau

La formula chiusa di Bernstein non è l'unico modo di valutare $B(t)$. Esiste un metodo geometrico equivalente, l'**algoritmo di de Casteljau**: si interpola linearmente e ripetutamente tra punti consecutivi (per la cubica: 3 interpolazioni lineari tra i 4 punti di controllo danno 3 punti intermedi, 2 interpolazioni tra questi danno 2 punti, un'ultima interpolazione dà $B(t)$). I due metodi producono lo stesso risultato, ma de Casteljau è numericamente più stabile per gradi alti (evita di calcolare esplicitamente $t^n$ e $(1-t)^n$, che perdono precisione quando $n$ cresce). Per il grado 3 di questo notebook la formula chiusa di Bernstein è più che adeguata, quindi è quella usata qui.

### Import e librerie

Importiamo `numpy` per il calcolo vettoriale e `matplotlib.pyplot` per i grafici. Le quattro funzioni riutilizzabili di questo notebook (`bezier_cubic`, `bezier_cubic_d1`, `bezier_cubic_d2`, `curvature`) non sono definite qui: vivono nel modulo condiviso `src/bezier.py` e vengono importate da lì, così da poter essere riusate — senza copiarne il codice — anche in altri notebook del progetto (in particolare, la stessa idea di curva di Bézier torna utile ogni volta che serve una parametrizzazione di forma con controllo locale).

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from src.bezier import bezier_cubic, bezier_cubic_d1, bezier_cubic_d2, curvature

### Verifica delle proprietà su un esempio

`bezier_cubic` è definita in `src/bezier.py` (stessa formula vista sopra) e importata nella prima cella. La testiamo con 4 punti di controllo scelti liberamente (non quelli del notebook della docente), verificando entrambe le proprietà dimostrate sopra:

- **Interpolazione agli estremi**: calcoliamo $B(0)-P_0$ e $B(1)-P_3$, che devono essere esattamente $(0,0)$.
- **Tangenza**: la tangente esatta in $t=0$ è $B'(0)=3(P_1-P_0)$, quindi parallela al segmento $P_0P_1$; approssimando la tangente con una differenza finita tra i primi due punti campionati della curva discretizzata, il test consiste nel verificare che questo vettore approssimato sia (quasi) parallelo al segmento di controllo. Per due vettori $a,b$ nel piano, il prodotto vettoriale (scalare, in 2D) è legato all'angolo $\theta$ tra loro da $a\times b = |a||b|\sin\theta$: se $a\times b \approx 0$ allora $\sin\theta\approx 0$, cioè $\theta\approx 0$ oppure $\theta\approx \pi$ — i due vettori sono (quasi) paralleli. È esattamente questo il test numerico usato per confermare la tangenza, sia in $t=0$ (contro il segmento $P_0P_1$) sia in $t=1$ (contro il segmento $P_2P_3$). Il piccolo scarto residuo dal valore teorico zero è dovuto al fatto che la tangente è approssimata con una differenza finita su una curva campionata con un numero finito di punti, non calcolata con la formula esatta $B'(t)$ (che verrà introdotta nello Step 3).

In [ ]:
P0 = (-2, -1)
P1 = (-1, 3)
P2 = (2, 4)
P3 = (4, 0)

curve, t = bezier_cubic(P0, P1, P2, P3)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(curve[:, 0], curve[:, 1], label="B(t)")
xs, ys = zip(P0, P1, P2, P3)
ax.plot(xs, ys, "o--", color="gray", label="poligono di controllo")
for P, name in zip((P0, P1, P2, P3), ("P0", "P1", "P2", "P3")):
    ax.annotate(name, P, textcoords="offset points", xytext=(5, 5))
ax.set_aspect("equal")
ax.legend()
ax.set_title("Bezier cubica: verifica interpolazione e tangenza agli estremi")
plt.show()

# Verifica numerica: interpolazione esatta agli estremi
print("B(0) - P0 =", curve[0] - np.asarray(P0, dtype=float))
print("B(1) - P3 =", curve[-1] - np.asarray(P3, dtype=float))

# Verifica numerica: tangenza (prodotto vettoriale 2D tra tangente discreta e segmento di controllo, atteso ~0)
tangent_start = curve[1] - curve[0]
segment_start = np.asarray(P1, dtype=float) - np.asarray(P0, dtype=float)
cross_start = tangent_start[0] * segment_start[1] - tangent_start[1] * segment_start[0]

tangent_end = curve[-1] - curve[-2]
segment_end = np.asarray(P3, dtype=float) - np.asarray(P2, dtype=float)
cross_end = tangent_end[0] * segment_end[1] - tangent_end[1] * segment_end[0]

print(f"prodotto vettoriale tangente/P0-P1 in t=0: {cross_start:.2e} (atteso ~0)")
print(f"prodotto vettoriale tangente/P2-P3 in t=1: {cross_end:.2e} (atteso ~0)")

## Step 2 — Profilo dello spoiler (dorso + ventre)

### Costruzione geometrica

Un profilo aerodinamico bidimensionale si ottiene con due curve che condividono lo stesso **bordo d'attacco** $P_0$ e lo stesso **bordo di uscita** $P_3$ — il segmento $P_0P_3$ è la **corda** del profilo — ma hanno punti di controllo intermedi diversi. Nel vocabolario aerodinamico, la superficie superiore si chiama **dorso** (o *estradosso*) e quella inferiore **ventre** (o *intradosso*); la curva a metà tra le due, punto per punto, è la **linea di camber** (o linea media), e la sua distanza dalla corda misura quanto il profilo è "curvo" nel suo insieme. Qui il dorso usa punti di controllo che spingono la curva in alto (profilo convesso, camber marcato), il ventre usa punti quasi piatti, appena sotto lo zero: le due curve, condividendo gli estremi, delimitano insieme una sagoma chiusa.

### Perché questa forma: dal profilo alare allo spoiler

Un profilo alare classico genera portanza perché il dorso è più convesso del ventre: una particella di fluido che percorre il dorso compie un tragitto più lungo, a parità di tempo di transito, rispetto a una che percorre il ventre, e quindi (nell'argomento classico, semplificato, basato sul teorema di Bernoulli lungo una linea di flusso in regime stazionario e non viscoso) vi transita più velocemente; una velocità locale maggiore corrisponde, per Bernoulli, a una pressione locale minore. La differenza di pressione tra dorso (bassa) e ventre (alta) produce una forza netta diretta dal ventre verso il dorso: la portanza.

> Va detto per completezza che questo argomento "a percorso più lungo" è la spiegazione qualitativa da manuale introduttivo, non la spiegazione rigorosa: la teoria completa della portanza richiede il concetto di **circolazione** attorno al profilo e la **condizione di Kutta** al bordo di uscita (che fissa in modo univoco la circolazione, altrimenti indeterminata per un flusso puramente potenziale). Per gli scopi qualitativi di questo notebook — e per la coerenza con il modello di flusso potenziale usato nello Step 4, che è anch'esso a circolazione nulla — l'argomento di Bernoulli è sufficiente a motivare la scelta geometrica.

Uno **spoiler** funziona secondo lo stesso principio ma capovolto nell'effetto voluto: l'obiettivo non è generare portanza (che solleverebbe il veicolo, riducendo l'aderenza), ma **deportanza**, cioè una forza diretta verso il basso che aumenta il carico sulle ruote e quindi la tenuta in curva ad alta velocità. Per questo motivo la sagoma qui costruita — dorso convesso in alto, ventre quasi piatto in basso — è pensata in modo che, montata sul veicolo, la superficie a maggiore curvatura (quella con velocità locale più alta e pressione più bassa) sia rivolta verso l'alto: la stessa asimmetria dorso/ventre che su un'ala genera portanza verso l'alto, orientata in questo modo genera una forza netta verso il basso.

In [ ]:
# Bordo d'attacco e bordo di uscita, condivisi da dorso e ventre
P0 = (0, 0)
P3 = (10, 0)

# Dorso: punti di controllo che spingono la curva in alto (profilo convesso)
P1_up = (2.5, 3.5)
P2_up = (7.5, 2.5)

# Ventre: punti di controllo quasi piatti, leggermente sotto lo zero
P1_low = (2.5, -0.4)
P2_low = (7.5, -0.3)

dorso, t_dorso = bezier_cubic(P0, P1_up, P2_up, P3)
ventre, t_ventre = bezier_cubic(P0, P1_low, P2_low, P3)

Plottiamo dorso e ventre sulla stessa figura, con $P_0$ e $P_3$ evidenziati, per vedere la sagoma completa. È importante che gli assi $x$ e $y$ usino la stessa scala (rapporto d'aspetto 1:1): con scale diverse, angoli e curvatura verrebbero distorti visivamente — un profilo con curvatura uniforme potrebbe sembrare più curvo in una direzione che nell'altra solo per effetto della scala del grafico, il che renderebbe fuorviante ogni valutazione qualitativa della forma (qui e, ancora di più, nello Step 3 quando si analizza la curvatura).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dorso[:, 0], dorso[:, 1], label="dorso", color="tab:blue")
ax.plot(ventre[:, 0], ventre[:, 1], label="ventre", color="tab:orange")
ax.plot(*P0, "ko")
ax.annotate("P0 (bordo d'attacco)", P0, textcoords="offset points", xytext=(0, -15))
ax.plot(*P3, "ko")
ax.annotate("P3 (bordo di uscita)", P3, textcoords="offset points", xytext=(-40, 8))
ax.set_aspect("equal")
ax.set_title("Profilo dello spoiler: dorso + ventre")
ax.legend()
plt.show()

## Step 3 — Curvatura lungo il profilo

### Derivate della Bézier

Applicando la regola generale vista nello Step 1 (la derivata di una Bézier di grado $n$ è una Bézier di grado $n-1$ sulle differenze successive dei punti di controllo) due volte, si ottengono le derivate prima e seconda della Bézier cubica, entrambe in forma chiusa:

$$B'(t) = 3(1-t)^2(P_1-P_0) + 6(1-t)t(P_2-P_1) + 3t^2(P_3-P_2),$$

$$B''(t) = 6(1-t)(P_2-2P_1+P_0) + 6t(P_3-2P_2+P_1).$$

La seconda si ottiene applicando la regola di derivazione alla Bézier di grado 2 che è $B'(t)/3$ (a meno del fattore $3$): le sue differenze successive sono $(P_2-P_1)-(P_1-P_0) = P_2-2P_1+P_0$ e $(P_3-P_2)-(P_2-P_1)=P_3-2P_2+P_1$, cioè le **differenze seconde** dei punti di controllo — un pattern che si ripete: la derivata $k$-esima di una Bézier di grado $n$ è (a meno di un fattore combinatorio) una Bézier di grado $n-k$ sulle differenze $k$-esime dei punti di controllo.

### Curvatura di una curva piana parametrica

Per una curva piana $r(t)=(x(t),y(t))$, la curvatura $\kappa(t)$ misura quanto rapidamente la direzione della tangente ruota rispetto alla **lunghezza d'arco** percorsa (non rispetto al parametro $t$, che è arbitrario). Sia $\varphi(t) = \mathrm{atan2}(y'(t), x'(t))$ l'angolo che la tangente forma con l'asse $x$, e sia $s(t)$ la lunghezza d'arco percorsa fino al tempo $t$, con $ds/dt = \|r'(t)\| = \sqrt{x'^2+y'^2}$. Per definizione:

$$\kappa(t) = \left|\frac{d\varphi}{ds}\right| = \left|\frac{d\varphi/dt}{ds/dt}\right|.$$

Da $\varphi = \mathrm{atan2}(y',x')$ si calcola $\dfrac{d\varphi}{dt} = \dfrac{x'y''-y'x''}{x'^2+y'^2}$ (derivata dell'arcotangente di un rapporto, con la regola della catena applicata a numeratore e denominatore che dipendono da $t$). Dividendo per $ds/dt=\sqrt{x'^2+y'^2}$ si ottiene la formula usata nel notebook:

$$\kappa(t) = \frac{|x'(t) y''(t) - y'(t) x''(t)|}{\left(x'(t)^2 + y'(t)^2\right)^{3/2}}.$$

**Interpretazione geometrica.** In ogni punto della curva esiste un unico cerchio — il *cerchio osculatore* — che approssima la curva al second'ordine in quel punto (stessa posizione, stessa tangente, stessa curvatura). Il suo raggio $\rho(t)$ è l'inverso della curvatura: $\rho(t) = 1/\kappa(t)$. Curvatura alta significa cerchio osculatore piccolo, cioè la curva "gira" strettamente; curvatura vicina a zero significa che la curva è quasi rettilinea in quel tratto.

**Perché il denominatore alla potenza $3/2$.** La curvatura è una proprietà **geometrica** della curva — dipende solo dalla sua forma nel piano, non da come viene percorsa. Se si cambia parametrizzazione (per esempio percorrendo la stessa curva il doppio più "veloce" in $t$), sia $x',y'$ sia $x'',y''$ cambiano, ma il rapporto che definisce $\kappa$ resta invariato esattamente perché il denominatore è la norma della velocità elevata al cubo: numeratore e denominatore scalano nello stesso modo (il numeratore è quadratico nelle derivate prime scalate per una seconda derivata anch'essa scalata; il conto degli esponenti torna) e l'effetto della velocità di percorrenza si cancella. Questo è ciò che rende $\kappa(t)$ una quantità sensata da confrontare punto per punto lungo la curva, indipendentemente da come è stata scelta la discretizzazione di $t$.

### Curvatura e rischio di separazione del flusso

Un tratto di profilo a curvatura elevata costringe il flusso a una forte accelerazione locale (per aggirare una curva stretta il fluido deve, in un modello di flusso potenziale, muoversi più rapidamente vicino alla parete). Per Bernoulli, un'accelerazione locale corrisponde a un calo di pressione; subito a valle di quel punto, quando la curvatura torna a diminuire, il flusso deve rallentare di nuovo, e la pressione risale: si genera così un **gradiente di pressione avverso** (pressione che cresce nella direzione del moto), che è il meccanismo fisico reale alla base della **separazione dello strato limite** — il flusso, rallentato dalla combinazione di attrito viscoso a parete e pressione crescente, può arrivare a invertire localmente il suo moto, staccandosi dalla superficie.

È importante essere precisi sui limiti di questo collegamento: la curvatura qui calcolata è un'euristica qualitativa, **non un criterio rigoroso di separazione**. La separazione reale dipende dal numero di Reynolds, dallo spessore e dal profilo di velocità dello strato limite viscoso, e dall'intensità e dall'estensione del gradiente di pressione avverso — nessuno di questi effetti è modellato qui (il modello di flusso dello Step 4, in particolare, è inviscido). Il punto di massima curvatura individuato sotto va quindi letto come un candidato plausibile per un punto critico, non come una predizione quantitativa.

`bezier_cubic_d1`, `bezier_cubic_d2` e `curvature` sono anch'esse definite in `src/bezier.py` (stesse formule sopra) e già importate. Calcoliamo $k(t)$ lungo il dorso, sulla stessa discretizzazione di $t$ usata per disegnarlo, e individuiamo il punto in cui la curvatura è massima.

In [ ]:
# Curvatura lungo il dorso, sullo stesso t usato per disegnarlo
k_dorso = curvature(P0, P1_up, P2_up, P3, t_dorso)
idx_max = np.argmax(k_dorso)

print(f"punto di massima curvatura: t = {t_dorso[idx_max]:.3f}, k = {k_dorso[idx_max]:.4f}")
print(f"coordinate sul dorso: {dorso[idx_max]}")

Marchiamo il punto di massima curvatura sia sul profilo sia sul grafico di $k(t)$. I due punti evidenziati coincidono necessariamente: entrambi sono individuati dallo stesso indice del massimo, calcolato una sola volta sulla discretizzazione di $t$ lungo il dorso — quell'indice seleziona sia la coppia $(x,y)$ sul profilo sia la coppia $(t,k(t))$ sul grafico della curvatura, quindi i due marker rappresentano necessariamente lo stesso punto fisico della curva, letto in due sistemi di coordinate diversi (spazio $(x,y)$ da un lato, spazio $(t,k)$ dall'altro).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Pannello sinistro: profilo con il punto di massima curvatura evidenziato
ax1.plot(dorso[:, 0], dorso[:, 1], label="dorso", color="tab:blue")
ax1.plot(ventre[:, 0], ventre[:, 1], label="ventre", color="tab:orange")
ax1.plot(*dorso[idx_max], "o", color="red", markersize=10, label="max curvatura", zorder=5)
ax1.set_aspect("equal")
ax1.set_title("Profilo con punto di massima curvatura")
ax1.legend()

# Pannello destro: k(t) lungo il dorso, stesso punto marcato
ax2.plot(t_dorso, k_dorso, color="tab:blue")
ax2.plot(t_dorso[idx_max], k_dorso[idx_max], "o", color="red", markersize=10, label="max curvatura", zorder=5)
ax2.set_xlabel("t")
ax2.set_ylabel("k(t)")
ax2.set_title("Curvatura lungo il dorso")
ax2.legend()

plt.tight_layout()
plt.show()

Il punto marcato nei due grafici è il tratto di dorso dove la curva "gira" più bruscamente, cioè dove il cerchio osculatore ha raggio minimo. Come discusso sopra, un tratto a curvatura elevata forza il flusso a un'accelerazione locale seguita da una decelerazione altrettanto brusca, generando il gradiente di pressione avverso che è il meccanismo fisico alla base della separazione dello strato limite: quel punto è quindi, qualitativamente, dove ci si aspetterebbe che un flusso reale abbia la maggiore probabilità di separarsi dalla superficie — un'indicazione utile per orientare la progettazione geometrica, non una predizione quantitativa (che richiederebbe un modello viscoso, non il flusso potenziale qualitativo dello Step 4).

## Step 4 — Flusso semplificato attorno allo spoiler

### Il potenziale complesso

Per un flusso bidimensionale, stazionario, incomprimibile e irrotazionale, esiste una funzione a valori complessi $w(z) = \varphi(x,y) + i\,\psi(x,y)$, detta **potenziale complesso**, dove $z=x+iy$, $\varphi$ è il potenziale di velocità e $\psi$ è la funzione di corrente; le linee $\psi=\text{cost}$ sono per costruzione le linee di flusso. Le componenti di velocità si ottengono da un'unica derivata complessa:

$$\frac{dw}{dz} = u - i v.$$

Questa identità (conseguenza delle equazioni di Cauchy-Riemann applicate a $\varphi$ e $\psi$) è il motivo per cui, in questa teoria, flussi complicati si costruiscono sommando pochi "mattoni" elementari con potenziale noto, e sommando le rispettive velocità.

### I due mattoni usati qui

- **Corrente uniforme** di velocità $U$ lungo $x$: $w_{\text{unif}}(z) = Uz$.
- **Doppietta** (dipolo di flusso) di intensità $\kappa$ nell'origine: $w_{\text{dip}}(z) = \kappa/z$.

Sommando le due, $w(z) = Uz + \kappa/z$, e scegliendo $\kappa = UR^2$, si ottiene la soluzione classica del flusso attorno a un **cilindro circolare di raggio $R$** centrato nell'origine:

$$w(z) = U\left(z + \frac{R^2}{z}\right).$$

Questa scelta di $\kappa$ non è arbitraria: si dimostra che con $\kappa=UR^2$ la circonferenza $|z|=R$ diventa essa stessa una linea di flusso ($\psi=\text{cost}$ su di essa), cioè il fluido non attraversa mai quella circonferenza — esattamente la condizione al contorno che deve valere sulla superficie di un ostacolo solido impenetrabile.

### Dalle formule complesse alle componenti $u,v$ implementate

Derivando:

$$\frac{dw}{dz} = U\left(1 - \frac{R^2}{z^2}\right).$$

Per estrarre $u,v$ separatamente serve la parte reale e immaginaria di $1/z^2$. Scrivendo $z = x+iy$, così che $|z|^2 = x^2+y^2 =: r^2$, si ha $1/z^2 = \bar z^2/|z|^4 = (x-iy)^2/r^4 = (x^2-y^2 - 2ixy)/r^4$, quindi

$$\frac{dw}{dz} = U\left(1 - \frac{R^2(x^2-y^2)}{r^4}\right) \;+\; i\,\frac{2UR^2xy}{r^4}.$$

Confrontando con $dw/dz = u - iv$ termine a termine (parte reale = $u$, parte immaginaria = $-v$):

$$u = U\left(1 - \frac{R^2(x^2-y^2)}{r^4}\right), \qquad v = -\frac{2UR^2xy}{r^4},$$

che sono esattamente le formule implementate sotto (con $x,y$ misurate a partire dal centro del cilindro equivalente, cioè `dx = X - xc`, `dy = Y - yc`, e `r2` $= r^2$).

### Cosa rappresenta, e cosa non rappresenta, questo modello

Questa soluzione descrive il flusso attorno a un **cilindro** — non attorno alla sagoma esatta dello spoiler costruita negli Step 1-2 — usato qui come "ostacolo equivalente" solo per farsi un'idea qualitativa di dove le linee di flusso si deformano maggiormente. Vanno tenuti presenti tre limiti, tutti conseguenze dirette delle ipotesi di partenza (flusso potenziale, cioè inviscido e irrotazionale):

1. **Circolazione nulla.** Questa soluzione non include alcuna circolazione attorno al cilindro (nessun termine logaritmico $i\Gamma \ln z /(2\pi)$ nel potenziale complesso). Per il **paradosso di d'Alembert**, un cilindro investito da un flusso potenziale a circolazione nulla non subisce alcuna forza netta (né portanza né resistenza): il modello qui usato è puramente qualitativo/visivo, non calcola alcuna forza aerodinamica sullo spoiler.
2. **Nessuna viscosità.** Senza viscosità non esiste strato limite, quindi nessuna separazione reale del flusso: la connessione con la curvatura discussa nello Step 3 resta un'euristica separata, non qualcosa che questo modello di flusso possa mostrare direttamente.
3. **Geometria approssimata.** Il cerchio è solo una sagoma "contenitore" di dimensione comparabile al profilo reale, non una rappresentazione accurata della sua forma.

In [ ]:
# Parametri del cilindro equivalente: centrato sul profilo, abbastanza grande da contenerlo
U = 1.0
xc, yc = 5.0, 0.9
R = 5.3

Costruiamo un reticolo di punti $(x,y)$ nell'area attorno allo spoiler e vi valutiamo il campo di velocità $(u,v)$ appena derivato. All'interno del cilindro equivalente (dove $r^2=(x-x_c)^2+(y-y_c)^2 < R^2$) il campo non ha senso fisico — quella regione rappresenta il "solido" attorno a cui il flusso scorre, non il flusso stesso — quindi quei punti vengono esclusi dal disegno delle linee di flusso nella cella successiva.

In [ ]:
# Griglia attorno allo spoiler
x_grid = np.linspace(xc - 1.6 * R, xc + 1.6 * R, 300)
y_grid = np.linspace(yc - 1.3 * R, yc + 1.3 * R, 300)
X, Y = np.meshgrid(x_grid, y_grid)

dx = X - xc
dy = Y - yc
r2 = dx ** 2 + dy ** 2

# Flusso potenziale: corrente uniforme + doppietta
u = U * (1 - R ** 2 * (dx ** 2 - dy ** 2) / r2 ** 2)
v = -U * (2 * R ** 2 * dx * dy / r2 ** 2)

# Non disegnare streamline dentro al cilindro (coperto comunque dalla sagoma piena)
inside = r2 < R ** 2
u = np.where(inside, np.nan, u)
v = np.where(inside, np.nan, v)

Disegniamo le linee di flusso insieme alla sagoma piena dello spoiler. Una **linea di flusso** è, per definizione, una curva che in ogni suo punto è tangente al vettore velocità locale $(u,v)$: è il percorso che seguirebbe una particella di fluido trasportata dal campo di velocità in quell'istante. La loro deformazione attorno all'ostacolo è appunto la visualizzazione qualitativa cercata in questo step.

La sagoma piena viene disegnata come un unico poligono chiuso ottenuto concatenando i punti del dorso con quelli del ventre percorsi **in ordine inverso**: partendo da $P_0$, il dorso viene percorso fino a $P_3$, poi il ventre viene ripercorso a ritroso da $P_3$ fino a $P_0$. Se il ventre venisse concatenato nello stesso verso del dorso, il poligono risultante attraverserebbe la figura da un lato all'altro invece di seguirne il contorno, producendo una forma a "farfalla" con un'autointersezione anziché la sagoma piena e chiusa desiderata.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.streamplot(X, Y, u, v, density=1.5, color="tab:blue", linewidth=0.8)

# Sagoma piena dello spoiler: dorso + ventre percorso all'indietro, poligono chiuso
poly_x = np.concatenate([dorso[:, 0], ventre[::-1, 0]])
poly_y = np.concatenate([dorso[:, 1], ventre[::-1, 1]])
ax.fill(poly_x, poly_y, color="black", zorder=5)

ax.set_aspect("equal")
ax.set_title("Flusso semplificato (cilindro equivalente) attorno allo spoiler")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()